# 스윙 콘솔 백테스트 — Colab 실행

PC에 아무것도 설치하지 않고 브라우저에서 백테스트를 돌린다.

**순서대로 셀을 실행한다.** 2번(빠른 점검)이 통과해야 3번(본 실행)으로 넘어간다.

| 셀 | 하는 일 | 걸리는 시간 |
|---|---|---|
| 0 | (선택) 구글 드라이브에 캐시 저장 — 세션이 끊겨도 재개 | 10초 |
| 1 | 저장소 클론 + 패키지 설치 | 1~2분 |
| 2 | 빠른 점검 (30종목·2년) | 2~5분 |
| 3 | 미장 본 실행 | 30분~1시간 |
| 4 | 국장 본 실행 | 1~2시간 |
| 5 | 리포트 확인 + 다운로드 | 즉시 |

> Colab 무료 세션은 90분 무활동 / 최대 12시간에서 끊긴다. 0번 셀로 캐시를
> 드라이브에 두면 끊겨도 받아둔 시세는 남으므로 재실행이 훨씬 빠르다.

In [ ]:
#@title 0. (선택) 구글 드라이브에 캐시 저장 — 세션이 끊겨도 재개
USE_DRIVE = True  #@param {type:"boolean"}

import os
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs('/content/drive/MyDrive/swing_backtest_cache', exist_ok=True)
    print('캐시 위치: /content/drive/MyDrive/swing_backtest_cache')
else:
    print('드라이브 미사용 — 세션이 끊기면 받은 시세도 사라진다')

In [ ]:
#@title 1. 저장소 클론 + 패키지 설치
BRANCH = "main"  #@param {type:"string"}
#@markdown 저장소가 비공개면 GitHub Personal Access Token 을 넣는다 (repo 읽기 권한).
#@markdown 공개면 비워둔다.
TOKEN = ""  #@param {type:"string"}

import os, subprocess, shutil

REPO = 'milianjw8-hue/jwcha-stock'
url = f'https://{TOKEN}@github.com/{REPO}.git' if TOKEN else f'https://github.com/{REPO}.git'

if os.path.exists('/content/jwcha-stock'):
    subprocess.run(['git', '-C', '/content/jwcha-stock', 'fetch', 'origin', BRANCH], check=False)
    subprocess.run(['git', '-C', '/content/jwcha-stock', 'checkout', BRANCH], check=False)
    subprocess.run(['git', '-C', '/content/jwcha-stock', 'pull', 'origin', BRANCH], check=False)
else:
    r = subprocess.run(['git', 'clone', '--branch', BRANCH, '--depth', '50', url, '/content/jwcha-stock'],
                       capture_output=True, text=True)
    if r.returncode:
        print(r.stderr[-800:])
        raise SystemExit('클론 실패 — 비공개 저장소면 TOKEN 을 넣거나, '
                         'GitHub 에서 브랜치 ZIP 을 받아 /content 에 업로드한다')

os.chdir('/content/jwcha-stock')

# 드라이브 캐시를 backtest/cache 에 연결 (심볼릭 링크)
DRIVE_CACHE = '/content/drive/MyDrive/swing_backtest_cache'
if os.path.isdir(DRIVE_CACHE):
    if os.path.islink('backtest/cache'):
        os.unlink('backtest/cache')
    elif os.path.isdir('backtest/cache'):
        shutil.rmtree('backtest/cache')
    os.symlink(DRIVE_CACHE, 'backtest/cache')
    print('캐시를 드라이브에 연결했다')

print(subprocess.run(['git', 'log', '--oneline', '-1'], capture_output=True, text=True).stdout)
!pip -q install -r scanner/requirements.txt
print('설치 완료')

In [ ]:
#@title 2. 빠른 점검 — 30종목·2년. 여기서 실패하면 본 실행은 시간 낭비다.
!python backtest/run.py --market us --quick

In [ ]:
#@title 3. 미장 본 실행 (30분~1시간)
START = "2019-01-01"  #@param {type:"string"}
SPLIT = "2024-01-01"  #@param {type:"string"}
NO_SWEEP = False  #@param {type:"boolean"}

cmd = f'python backtest/run.py --market us --start {START} --split {SPLIT}'
if NO_SWEEP:
    cmd += ' --no-sweep'
print(cmd)
!{cmd}

In [ ]:
#@title 4. 국장 본 실행 (1~2시간) — KRX 는 종목당 순차 조회라 느리다
START_KR = "2019-01-01"  #@param {type:"string"}
SPLIT_KR = "2024-01-01"  #@param {type:"string"}
UNIVERSE = 150  #@param {type:"integer"}

!python backtest/run.py --market kr --start {START_KR} --split {SPLIT_KR} --universe {UNIVERSE}

In [ ]:
#@title 5. 리포트 확인 + 다운로드
import os, json
from IPython.display import Markdown, display

for mkt in ('us', 'kr'):
    p = f'docs/backtest_{mkt}.md'
    if os.path.exists(p):
        display(Markdown(open(p, encoding='utf-8').read()))
    else:
        print(f'{p} 없음 — 해당 시장은 아직 실행하지 않았다')

# 파일로 내려받기
try:
    from google.colab import files
    for mkt in ('us', 'kr'):
        for ext in ('md', 'json'):
            p = f'docs/backtest_{mkt}.{ext}'
            if os.path.exists(p):
                files.download(p)
except Exception as e:
    print('다운로드 건너뜀:', e)

## 결과를 저장소에 올리려면 (선택)

리포트를 브랜치에 커밋해두면 앱 일지 탭의 '룰셋 검증' 카드에 자동으로 표시된다.
쓰기 권한이 있는 TOKEN 이 필요하다.

```python
!git config user.email "you@example.com"
!git config user.name "me"
!git add docs/backtest_*.json docs/backtest_*.md
!git commit -m "chore: 백테스트 리포트"
!git push origin HEAD
```

올리지 않고 `.md` 파일 내용을 그대로 붙여넣어 공유해도 된다.